# KATA Environment Demo

This notebook demonstrates how to use the **KataEnv** Gymnasium environment for
knowledge-aware technician assignment in a simulated manufacturing plant.

We cover:
1. Building a scenario from config
2. Running the environment with **structured** observations
3. Running with **token** observations
4. Running with **token_ids** (integer-encoded, Transformer-ready)
5. Using the **MCA warmup** for encoder fitting
6. Customising reward components
7. Writing a simple heuristic agent loop

In [ ]:
import os, sys, warnings, logging

# Suppress noisy simulation logs for cleaner output
logging.disable(logging.WARNING)
warnings.filterwarnings("ignore")

# Ensure we don't load an external config file
os.environ["KATA_CONF_PATH"] = "/dev/null/no_such_file.json"

# Context manager to silence SimPy print() calls from machines/routers/etc.
from contextlib import contextmanager

@contextmanager
def quiet():
    """Suppress stdout from the simulation (machine/router/source prints)."""
    old = sys.stdout
    sys.stdout = open(os.devnull, "w")
    try:
        yield
    finally:
        sys.stdout.close()
        sys.stdout = old

## 1. Define a factory scenario

All simulation parameters live in a single `KATAConfig` object.
Entity-level configs (`TechnicianConfig`, `MachineConfig`, etc.) are nested inside it.
The `ScenarioBuilder` turns a config into a `(simpy.Environment, GymTechDispatcher)` pair.

In [ ]:
from kata.core.config import (
    KATAConfig,
    GymEnvConfig,
    TechnicianConfig,
    MachineConfig,
    ProductConfig,
    ComponentConfig,
)
from kata.scenario import ScenarioBuilder

cfg = KATAConfig(
    technicians={
        "expert": TechnicianConfig(
            name="expert",
            fatigue_lambda=0.005,   # slow fatigue build-up
            knowledge_learning_rate=0.15,
        ),
        "junior_1": TechnicianConfig(
            name="junior_1",
            fatigue_lambda=0.02,    # tires faster
            knowledge_learning_rate=0.05,
        ),
        "junior_2": TechnicianConfig(
            name="junior_2",
            fatigue_lambda=0.02,
            knowledge_learning_rate=0.05,
        ),
    },
    machines={
        "cnc": MachineConfig(
            machine_type="CNC",
            process_time=100,
            components={
                "spindle": ComponentConfig(
                    component_id="spindle_0",
                    component_type="spindle",
                    base_repair_time=60.0,
                ),
            },
        ),
        "assembly": MachineConfig(
            machine_type="Assembly",
            process_time=80,
        ),
    },
    products={
        "widget": ProductConfig(route=["CNC", "Assembly"]),
    },
    gym=GymEnvConfig(
        max_episode_steps=100,
        max_sim_time=5000.0,
    ),
)

print(f"Technicians: {list(cfg.technicians.keys())}")
print(f"Machines:    {list(cfg.machines.keys())}")
print(f"Products:    {list(cfg.products.keys())}")

## 2. Structured observations (default)

The default `observation_representation="structured"` returns a dict of numpy arrays:
- `sim_time` - current simulation clock
- `has_open_ticket` - whether there is a pending repair request
- `ticket_created_at` / `ticket_machine_id` - details of the current ticket
- `technician_busy` - binary vector of technician availability
- `technician_fatigue` - float vector (if enabled)
- `pending_queue_size` - number of queued tickets (if enabled)

In [ ]:
from kata.env import KataEnv

def make_factory():
    return ScenarioBuilder(cfg).build()

with quiet():
    env = KataEnv(scenario_factory=make_factory, config=cfg.gym)
    obs, info = env.reset()

print("Observation space:", env.observation_space)
print("Action space:     ", env.action_space)
print()
for key, val in obs.items():
    print(f"  {key:25s} = {val}")
print()
print("Info:", info)

### Run a few steps with random actions

In [ ]:
total_reward = 0.0
with quiet():
    for step in range(20):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break

print(f"Ran {step+1} steps")
print(f"Total reward: {total_reward:.3f}")
print(f"Final sim time: {info['sim_time']:.0f}")

## 3. Token observations

Set `observation_representation="tokens"` to get string-valued token tuples.
Each observation field is emitted as a **key token** (e.g. `MACHINE_BROKEN`)
followed by a **value token** (e.g. `TRUE`, `C_4_5`, `T_500_1K`).

Value tokens use bucketing to keep the vocabulary bounded:
- **Time** values: `T_0_50`, `T_50_200`, `T_200_500`, `T_500_1K`, `T_1K_5K`, `T_5K+`
- **Ratios** (fatigue, knowledge): `R_0`, `R_LOW`, `R_MEDLOW`, `R_MED`, `R_MEDHIGH`, `R_HIGH`
- **Counts**: `C_0`, `C_1`, `C_2_3`, `C_4_5`, `C_6_10`, `C_11_20`, `C_20+`
- **Booleans**: `TRUE` / `FALSE`
- **Categorical**: literal strings (`CNC`, `Assembly`, `ticket_only`, ...)

In [ ]:
token_cfg = cfg.gym.model_copy(update={
    "observation_representation": "tokens",
    "observation_mode": "factory_level",
    "include_technician_fatigue_tokens": True,
    "include_technician_knowledge_tokens": True,
    "token_observation_length": 32,
})

with quiet():
    env_tok = KataEnv(scenario_factory=make_factory, config=token_cfg)
    obs, _ = env_tok.reset()

print("Token observation (first 20 tokens):")
for i, tok in enumerate(obs["tokens"][:20]):
    print(f"  [{i:2d}] {tok}")

## 4. Token ID observations (Transformer-ready)

Set `observation_representation="token_ids"` to get integer-encoded sequences.
Each unique token string is mapped to a unique integer via the `StateTokenizer`.
The output is a fixed-length `int64` array, padded with `0` (`<PAD>`).

In [ ]:
tid_cfg = cfg.gym.model_copy(update={
    "observation_representation": "token_ids",
    "tokenizer_seq_length": 32,
})

with quiet():
    env_ids = KataEnv(scenario_factory=make_factory, config=tid_cfg)
    obs, _ = env_ids.reset()

print(f"token_ids shape: {obs['token_ids'].shape}")
print(f"token_ids dtype: {obs['token_ids'].dtype}")
print(f"token_ids:       {obs['token_ids']}")
print(f"\nVocab size: {env_ids._tokenizer.vocab_size}")

## 5. MCA warmup

When `use_mca_encoder=True`, the first call to `reset()` runs a warmup phase:
1. A `WarmupEnv` runs the simulation under a heuristic policy (least-busy tech)
2. Repair requests are collected and used to fit a `prince.MCA` encoder
3. The tokenizer vocabulary is pre-populated from observed tokens
4. The fitted MCA encoder becomes the global encoder for knowledge updates

This is useful when you want repair request features (machine type, component type, etc.)
to be embedded via MCA rather than simple hashing.

In [ ]:
mca_cfg = cfg.gym.model_copy(update={
    "observation_representation": "token_ids",
    "use_mca_encoder": True,
    "warmup_steps": 50,
    "mca_grid_shape": (10, 10),
    "mca_n_components": 2,
    "tokenizer_seq_length": 32,
})

with quiet():
    env_mca = KataEnv(scenario_factory=make_factory, config=mca_cfg)
    obs, info = env_mca.reset()

print(f"Warmup done:    {env_mca._warmup_done}")
print(f"MCA fitted:     {env_mca._mca_encoder.fitted if env_mca._mca_encoder else False}")
print(f"Vocab size:     {env_mca._tokenizer.vocab_size}")
print(f"Vocab frozen:   {env_mca._tokenizer.frozen}")
print(f"token_ids:      {obs['token_ids']}")

## 6. Customising reward components

The reward is a weighted sum of independent components, each toggled and scaled via config.
The `info` dict includes a `reward_breakdown` showing each component's contribution.

In [ ]:
from kata.core.config import GymRewardConfig, RewardComponentConfig

reward_cfg = cfg.gym.model_copy(update={
    "reward": GymRewardConfig(
        assignment=RewardComponentConfig(enabled=True, coefficient=1.0),
        wait_time=RewardComponentConfig(enabled=True, coefficient=2.0),
        queue_size=RewardComponentConfig(enabled=True, coefficient=0.5),
        busy_technician=RewardComponentConfig(enabled=True, coefficient=3.0),
    ),
    "assignment_reward": 1.0,
})

with quiet():
    env_r = KataEnv(scenario_factory=make_factory, config=reward_cfg)
    obs, _ = env_r.reset()
    results = []
    for step in range(5):
        action = env_r.action_space.sample()
        obs, reward, terminated, truncated, info = env_r.step(action)
        results.append((step, reward, info["reward_breakdown"]))
        if terminated:
            break

for step, reward, breakdown in results:
    print(f"Step {step} | reward={reward:+.3f} | breakdown={breakdown}")

## 7. Heuristic agent loop

A simple baseline: always assign to the **least fatigued, non-busy** technician.

In [ ]:
import numpy as np

heuristic_cfg = cfg.gym.model_copy(update={
    "max_episode_steps": 200,
    "max_sim_time": 10_000.0,
})


def least_fatigued_action(obs):
    """Pick the non-busy tech with lowest fatigue. Fall back to random."""
    busy = obs["technician_busy"]
    fatigue = obs.get("technician_fatigue", np.zeros_like(busy, dtype=np.float32))
    # Penalise busy techs
    scores = fatigue + busy * 100.0
    return int(np.argmin(scores))


with quiet():
    env_h = KataEnv(scenario_factory=make_factory, config=heuristic_cfg)
    obs, _ = env_h.reset()
    rewards = []
    for _ in range(200):
        action = least_fatigued_action(obs)
        obs, reward, terminated, truncated, info = env_h.step(action)
        rewards.append(reward)
        if terminated or truncated:
            break

print(f"Episode length: {len(rewards)} steps")
print(f"Total reward:   {sum(rewards):.3f}")
print(f"Mean reward:    {np.mean(rewards):.4f}")
print(f"Final sim time: {info['sim_time']:.0f}")

## 8. Using a JSON config file

Instead of building the config in Python, you can point to a JSON file.
Pre-built configs are available in `run_configs/`:
- `config.json` - default single-machine setup
- `complex_factory.json` - multi-machine factory with components
- `minimal.json` - minimal setup for quick tests

```python
import os
os.environ["KATA_CONF_PATH"] = "run_configs/complex_factory.json"

from kata.core.config import KATAConfig
cfg = KATAConfig()  # loads from JSON
```

## Summary

| Feature | Config key | Values |
|---|---|---|
| Observation format | `observation_representation` | `"structured"`, `"tokens"`, `"token_ids"` |
| Observation detail | `observation_mode` | `"ticket_only"`, `"broken_machine"`, `"factory_level"` |
| MCA encoder warmup | `use_mca_encoder` | `True` / `False` |
| Reward shaping | `reward.*` | Per-component `enabled` + `coefficient` |
| Tokenizer length | `tokenizer_seq_length` | int (default 64) |